In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("DATA\oficial.csv")

<>:1: SyntaxWarning: invalid escape sequence '\o'
<>:1: SyntaxWarning: invalid escape sequence '\o'
C:\Users\pedro\AppData\Local\Temp\ipykernel_508\3862914280.py:1: SyntaxWarning: invalid escape sequence '\o'
  df = pd.read_csv("DATA\oficial.csv")


In [3]:
df.head()

,temporada_atual,rodada_atual,id_piloto_atual,posicao_quali_atual,q1_atual,q2_atual,q3_atual,target,id_circuito_atual,id_equipe_atual,...,media_ultimas_3_anterior,media_ultimas_5_anterior,qtde_abandonos_anterior,media_posicao_ganha_anterior,tendencia_desempenho,temp_ar_media,temp_pista_media,umidade_media,corrida_molhada,perc_voltas_chuva
0,2018,1,alonso,11.0,83597.0,83692.0,NaN,5,albert_park,mclaren,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
1,2018,1,bottas,10.0,83686.0,82089.0,NaN,8,albert_park,mercedes,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
2,2018,1,brendon_hartley,16.0,84532.0,NaN,NaN,15,albert_park,toro_rosso,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
3,2018,1,ericsson,17.0,84556.0,NaN,NaN,19,albert_park,sauber,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
4,2018,1,gasly,20.0,85295.0,NaN,NaN,18,albert_park,toro_rosso,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5


In [4]:
print(df.columns)

Index(['temporada_atual', 'rodada_atual', 'id_piloto_atual',
       'posicao_quali_atual', 'q1_atual', 'q2_atual', 'q3_atual', 'target',
       'id_circuito_atual', 'id_equipe_atual', 'grid_anterior',
       'posicao_ultima_corrida', 'posicao_equipe_anterior',
       'pontos_equipe_anterior', 'vitorias_equipe_anterior',
       'dif_para_pole_atual', 'pontos_anterior', 'posicao_camp_anterior',
       'num_vitorias_anterior', 'media_ultimas_3_anterior',
       'media_ultimas_5_anterior', 'qtde_abandonos_anterior',
       'media_posicao_ganha_anterior', 'tendencia_desempenho', 'temp_ar_media',
       'temp_pista_media', 'umidade_media', 'corrida_molhada',
       'perc_voltas_chuva'],
      dtype='object')


Info antes da quali

In [ ]:
from datetime import datetime
import requests

ANO_ATUAL = datetime.now().year
proxima_rodada = df[(df["temporada_atual"] == ANO_ATUAL)]["rodada_atual"].max()
pilotos = df[(df["temporada_atual"] == ANO_ATUAL) & (df["rodada_atual"] == proxima_rodada)]["id_piloto_atual"]
circuito = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada + 1}/circuits/").json()["MRData"]["CircuitTable"]["Circuits"][0]["circuitId"]
equipe = df[(df["temporada_atual"] == ANO_ATUAL) & (df["rodada_atual"] == proxima_rodada)]["id_equipe_atual"]

grid = []
infos_equipe = []

acesso_results = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada}/results/").json()
results = acesso_results["MRData"]["RaceTable"]["Races"][0]["Results"]
for result in results:
    grid.append({
        "temporada_atual": int(acesso_results["MRData"]["RaceTable"]["season"]),
        "rodada_atual": int(acesso_results["MRData"]["RaceTable"]["round"]),
        "id_piloto_atual": result["Driver"]["driverId"],
        "grid_anterior": result["grid"],
        "posicao_ultima_corrida": result["position"]
    })
df_grid = pd.DataFrame(grid)

acesso_equipe = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada}/constructorstandings/").json()
teams = acesso_equipe["MRData"]["StandingsTable"]["StandingsLists"][0]["ConstructorStandings"]
for team in teams:
    infos_equipe.append({
        "temporada_atual": int(acesso_equipe["MRData"]["StandingsTable"]["season"]),
        "rodada_atual": int(acesso_equipe["MRData"]["StandingsTable"]["round"]),
        "id_equipe_atual": team["Constructor"]["constructorId"],
        "posicao_equipe_anterior": team["position"],
        "pontos_equipe_anterior": team["points"],
        "vitorias_equipe_anterior": team["wins"]
    })
df_team = pd.DataFrame(infos_equipe)

df = pd.DataFrame({
    'temporada_atual': ANO_ATUAL,
    'rodada_atual': proxima_rodada,
    'id_piloto_atual': pilotos,
    "id_circuito_atual": circuito,
    'id_equipe_atual': equipe,
})

In [15]:
df_merge1 = pd.merge(
    df,
    df_grid,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

df_merge1.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida
0,2026,6,albon,catalunya,williams,11,8
1,2026,6,alonso,catalunya,aston_martin,21,10
2,2026,6,antonelli,catalunya,mercedes,1,1
3,2026,6,arvid_lindblad,catalunya,rb,15,6
4,2026,6,bearman,catalunya,haas,19,20


In [16]:
df_merge2 = pd.merge(
    df_merge1,
    df_team,
    on=["temporada_atual", "rodada_atual", "id_equipe_atual"],
    how="left"
)

df_merge2.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,posicao_equipe_anterior,pontos_equipe_anterior,vitorias_equipe_anterior
0,2026,6,albon,catalunya,williams,11,8,8,11,0
1,2026,6,alonso,catalunya,aston_martin,21,10,10,1,0
2,2026,6,antonelli,catalunya,mercedes,1,1,1,244,6
3,2026,6,arvid_lindblad,catalunya,rb,15,6,6,39,0
4,2026,6,bearman,catalunya,haas,19,20,7,21,0
